# Feature engineering

This notebook focuses on developing features for the model of predicting the sales potential of new locations.

 Planned features:

Competition:
- competitor_count_1500m — number of competitor locations in the 1500-meter radius
- nearest_competitor_m — distance to the nearest competitor

POI:
- poi_count_1500m — total number of POIs
- poi_[category]_count_1500m — total number of POIs per category
- nearest_poi_m - distance to the nearest POI
- nearest_poi_[category]_m - distance to the nearest category of POI

Buildings:
- building_count_1500m — number of buildings
- residential_ratio_1500m — ratio of residential buildings

Population:
- population_1500m — total number of residents

Footfall:
- signals_1500m — number of signals
- unique_users_1500m — unique users

The radius of 1500 meters was chosen empirically and can be changed by modifying the `RADIUS` variable below and rerunning the notebook.

In [1]:
import geopandas as gpd
from pathlib import Path

PROCESSED_PATH = Path("../data/processed")

gdf_locations    = gpd.read_parquet(PROCESSED_PATH / "gdf_locations.parquet")
gdf_competitors  = gpd.read_parquet(PROCESSED_PATH / "gdf_competitors.parquet")
gdf_poi          = gpd.read_parquet(PROCESSED_PATH / "gdf_poi.parquet")
gdf_area         = gpd.read_parquet(PROCESSED_PATH / "gdf_area.parquet")
gdf_districts    = gpd.read_parquet(PROCESSED_PATH / "gdf_districts.parquet")
gdf_buildings    = gpd.read_parquet(PROCESSED_PATH / "gdf_buildings.parquet")
gdf_population   = gpd.read_parquet(PROCESSED_PATH / "gdf_population.parquet")

In [2]:
RADIUS = 1500

Specific methods of computing each group of features is present in `src/features.py`. The exact process of computing each feature is described in the comments for each method.

In [3]:
from src.features import compute_competitor_features, compute_poi_features, compute_buildings_features, compute_population_features, compute_footfall_features

#### Competitors

In [4]:
competitor_features = compute_competitor_features(gdf_locations, gdf_competitors, RADIUS)

In [5]:
competitor_features

,location_id,competitor_count_1500m,nearest_competitor_m
0,LOC_001,1,262.795371
1,LOC_002,0,2567.458932
2,LOC_003,2,619.198805
3,LOC_004,0,9153.895298
4,LOC_005,1,1211.453292
5,LOC_006,0,2383.421789
6,LOC_007,0,2389.236196
7,LOC_008,0,3604.922988
8,LOC_009,0,2946.215401
9,LOC_010,0,1854.202830


In [6]:
print(competitor_features[f'competitor_count_{RADIUS}m'].value_counts())
competitor_features.info()
competitor_features.describe()

competitor_count_1500m
0    27
1    19
2     2
3     1
4     1
Name: count, dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 3 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   location_id             50 non-null     object 
 1   competitor_count_1500m  50 non-null     int64  
 2   nearest_competitor_m    50 non-null     float64
dtypes: float64(1), int64(1), object(1)
memory usage: 1.3+ KB


,competitor_count_1500m,nearest_competitor_m
count,50.000000,50.000000
mean,0.600000,2541.437919
std,0.832993,2867.428153
min,0.000000,42.173981
25%,0.000000,1047.760275
50%,0.000000,1711.156093
75%,1.000000,2892.203181
max,4.000000,13842.306595


#### POIs

In [7]:
poi_features = compute_poi_features(gdf_locations, gdf_poi, RADIUS)
poi_features

,location_id,poi_count_1500m,nearest_poi_m,poi_restaurant_count_1500m,nearest_poi_restaurant_m,poi_bank_count_1500m,nearest_poi_bank_m,poi_school_count_1500m,nearest_poi_school_m,poi_bus_stop_count_1500m,nearest_poi_bus_stop_m,poi_park_count_1500m,nearest_poi_park_m,poi_pharmacy_count_1500m,nearest_poi_pharmacy_m,poi_hospital_count_1500m,nearest_poi_hospital_m,poi_mall_count_1500m,nearest_poi_mall_m
0,LOC_001,0,1644.258581,0,3508.276856,0,4935.923724,0,5093.826731,0,1644.258581,0,2085.040228,0,5116.461759,0,6887.945370,0,10483.383593
1,LOC_002,11,442.687835,4,834.054653,0,7855.760495,3,442.687835,0,2139.914263,1,1264.122489,2,679.653896,1,1469.660936,0,7860.430423
2,LOC_003,2,176.808447,0,1924.769849,0,1564.633471,0,2262.431183,1,524.253413,0,2063.400799,0,2330.359427,0,2622.969514,1,176.808447
3,LOC_004,0,2041.842112,0,11708.981182,0,14548.715383,0,8757.348413,0,6385.916456,0,2041.842112,0,11188.412501,0,12702.138624,0,13938.294690
4,LOC_005,1,1081.644470,0,5302.691028,0,2048.529943,0,1955.442974,0,1586.395355,0,2026.654262,0,3828.897338,1,1081.644470,0,6566.012106
5,LOC_006,5,457.328429,1,1066.233527,0,5718.489810,1,457.328429,1,594.385991,0,3299.390423,0,6434.450495,2,856.805877,0,4091.387108
6,LOC_007,1,794.507330,0,1747.796648,0,2557.193166,0,1782.480673,0,3740.160704,0,4480.726139,0,2504.708304,1,794.507330,0,8035.228268
7,LOC_008,1,85.340871,0,3378.313139,1,85.340871,0,11357.258138,0,6772.501741,0,5000.899037,0,11866.487002,0,11582.301804,0,4696.770810
8,LOC_009,8,220.296106,2,1283.429958,0,7598.074758,4,558.759084,0,2337.700533,1,955.976278,1,220.296106,0,1926.974884,0,8238.469173
9,LOC_010,6,76.985326,0,3590.490489,1,110.580686,1,132.079638,2,721.354432,1,76.985326,0,2091.077229,1,922.700778,0,6850.431334


In [8]:
for cat in gdf_poi['category'].unique():
    print(poi_features[f'poi_{cat}_count_{RADIUS}m'].value_counts())
poi_features.info()
poi_features.describe()

poi_restaurant_count_1500m
0    34
1     7
4     4
2     4
6     1
Name: count, dtype: int64
poi_bank_count_1500m
0    36
1    12
3     1
2     1
Name: count, dtype: int64
poi_school_count_1500m
0    31
1    12
2     4
3     2
4     1
Name: count, dtype: int64
poi_bus_stop_count_1500m
0    43
1     6
2     1
Name: count, dtype: int64
poi_park_count_1500m
0    42
1     8
Name: count, dtype: int64
poi_pharmacy_count_1500m
0    38
1     8
2     4
Name: count, dtype: int64
poi_hospital_count_1500m
0    36
1    11
2     3
Name: count, dtype: int64
poi_mall_count_1500m
0    45
1     5
Name: count, dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 19 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   location_id                 50 non-null     object 
 1   poi_count_1500m             50 non-null     int64  
 2   nearest_poi_m               50 non-null     float64


,poi_count_1500m,nearest_poi_m,poi_restaurant_count_1500m,nearest_poi_restaurant_m,poi_bank_count_1500m,nearest_poi_bank_m,poi_school_count_1500m,nearest_poi_school_m,poi_bus_stop_count_1500m,nearest_poi_bus_stop_m,poi_park_count_1500m,nearest_poi_park_m,poi_pharmacy_count_1500m,nearest_poi_pharmacy_m,poi_hospital_count_1500m,nearest_poi_hospital_m,poi_mall_count_1500m,nearest_poi_mall_m
count,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.00000,50.000000,50.000000,50.000000,50.000000,50.000000
mean,2.760000,1143.567192,0.740000,4145.817302,0.340000,4506.965995,0.600000,3096.028295,0.160000,3275.748776,0.160000,4346.303301,0.32000,4298.892569,0.340000,5090.687339,0.100000,6621.682116
std,3.178756,939.156015,1.396935,4821.563298,0.626295,3726.922711,0.947607,3563.147009,0.421852,2325.369397,0.370328,3952.582690,0.62073,3676.879775,0.592814,5337.100689,0.303046,4390.363725
min,0.000000,68.978531,0.000000,203.850066,0.000000,85.340871,0.000000,132.079638,0.000000,524.253413,0.000000,76.985326,0.00000,200.096943,0.000000,361.858858,0.000000,68.978531
25%,0.000000,446.347983,0.000000,1283.763264,0.000000,1476.223291,0.000000,947.272368,0.000000,1919.604065,0.000000,2026.625667,0.00000,1541.225431,0.000000,1407.362191,0.000000,3362.850426
50%,1.000000,1088.599208,0.000000,2848.802858,0.000000,3738.326257,0.000000,1944.656686,0.000000,2720.164788,0.000000,3309.618747,0.00000,3310.745654,0.000000,3465.425485,0.000000,6489.027542
75%,4.750000,1549.920838,1.000000,4518.990258,1.000000,7307.655967,1.000000,4033.345869,0.000000,3669.866345,0.000000,4640.146889,0.00000,6096.923407,1.000000,5874.092827,0.000000,8437.805672
max,11.000000,4023.653180,6.000000,20968.450341,3.000000,14607.955095,4.000000,20276.500680,2.000000,12713.799657,1.000000,20524.838522,2.00000,13094.836681,2.000000,20515.138507,1.000000,18864.533654


#### Buildings

In [9]:
buildings_features = compute_buildings_features(gdf_locations, gdf_buildings, RADIUS)

In [10]:
buildings_features

,location_id,building_count_1500m,residential_ratio_1500m
0,LOC_001,1345,0.591822
1,LOC_002,4861,0.565110
2,LOC_003,2737,0.471319
3,LOC_004,1784,0.613789
4,LOC_005,2437,0.499795
5,LOC_006,2640,0.628788
6,LOC_007,2818,0.521292
7,LOC_008,1949,0.632632
8,LOC_009,4801,0.605082
9,LOC_010,4910,0.558045


In [11]:
buildings_features.info()
buildings_features.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 3 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   location_id              50 non-null     object 
 1   building_count_1500m     50 non-null     int64  
 2   residential_ratio_1500m  50 non-null     float64
dtypes: float64(1), int64(1), object(1)
memory usage: 1.3+ KB


,building_count_1500m,residential_ratio_1500m
count,50.000000,50.000000
mean,2861.380000,0.589015
std,1303.197307,0.080394
min,621.000000,0.294686
25%,1928.750000,0.545999
50%,2769.000000,0.593963
75%,3684.750000,0.629485
max,5824.000000,0.736969


#### Population

In [12]:
population_features = compute_population_features(gdf_locations, gdf_population, RADIUS)

In [13]:
population_features

,location_id,population_1500m
0,LOC_001,15892
1,LOC_002,37983
2,LOC_003,18939
3,LOC_004,3574
4,LOC_005,30322
5,LOC_006,37003
6,LOC_007,23388
7,LOC_008,8781
8,LOC_009,34019
9,LOC_010,42146


In [14]:
population_features.info()
population_features.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   location_id       50 non-null     object
 1   population_1500m  50 non-null     int64 
dtypes: int64(1), object(1)
memory usage: 928.0+ bytes


,population_1500m
count,50.000000
mean,22990.320000
std,15308.202973
min,1968.000000
25%,9105.000000
50%,21722.500000
75%,34405.250000
max,59635.000000


#### Footfall

In [15]:
import json

with open('../data/processed/analysis_bbox.json') as f:
    ANALYSIS_BBOX = json.load(f)

In [16]:
BBOX_FILTER = f"""
    CAST(latitude AS FLOAT) BETWEEN {ANALYSIS_BBOX['lat_min']} AND {ANALYSIS_BBOX['lat_max']}
    AND CAST(longitude AS FLOAT) BETWEEN {ANALYSIS_BBOX['lng_min']} AND {ANALYSIS_BBOX['lng_max']}
"""

DATE_FILTER = """
    YEAR(TRY_TO_TIMESTAMP(occured_at)) = 2020
    AND MONTH(TRY_TO_TIMESTAMP(occured_at)) = 7
"""

BASE_FILTER = f"""
    {BBOX_FILTER} AND {DATE_FILTER}
"""

In [17]:
import pandas as pd

FOOTFALL_CACHE = Path('../data/processed/footfall_features.parquet')

if FOOTFALL_CACHE.exists():
    footfall_features = pd.read_parquet(FOOTFALL_CACHE)
    print("Loaded from cache")
else:
    footfall_features = compute_footfall_features(gdf_locations, BASE_FILTER, RADIUS)
    footfall_features.to_parquet(FOOTFALL_CACHE)
    print("Computed and cached")

Computed and cached


In [18]:
footfall_features

,location_id,signals_1500m,unique_users_1500m
0,LOC_019,225231,3349
1,LOC_037,2572,163
2,LOC_045,203124,2059
3,LOC_014,48558,934
4,LOC_033,47229,996
5,LOC_020,84976,1433
6,LOC_013,116090,1057
7,LOC_040,55600,1010
8,LOC_002,224210,2366
9,LOC_025,50988,972


In [19]:
footfall_features.info()
footfall_features.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 3 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   location_id         50 non-null     object
 1   signals_1500m       50 non-null     int64 
 2   unique_users_1500m  50 non-null     int64 
dtypes: int64(2), object(1)
memory usage: 1.3+ KB


,signals_1500m,unique_users_1500m
count,50.000000,50.000000
mean,87122.660000,1222.580000
std,65478.587593,745.034997
min,2572.000000,159.000000
25%,33297.750000,568.250000
50%,73094.000000,1224.500000
75%,138841.250000,1631.250000
max,225231.000000,3349.000000


In [20]:
features_df = (
    gdf_locations[['location_id', 'lat', 'lng', 'monthly_revenue']]
    .merge(competitor_features, on='location_id', how='left')
    .merge(poi_features,        on='location_id', how='left')
    .merge(buildings_features,   on='location_id', how='left')
    .merge(population_features, on='location_id', how='left')
    .merge(footfall_features,   on='location_id', how='left')
)

print(f"Shape: {features_df.shape}")
print(features_df.isna().sum())

Shape: (50, 29)
location_id                   0
lat                           0
lng                           0
monthly_revenue               0
competitor_count_1500m        0
nearest_competitor_m          0
poi_count_1500m               0
nearest_poi_m                 0
poi_restaurant_count_1500m    0
nearest_poi_restaurant_m      0
poi_bank_count_1500m          0
nearest_poi_bank_m            0
poi_school_count_1500m        0
nearest_poi_school_m          0
poi_bus_stop_count_1500m      0
nearest_poi_bus_stop_m        0
poi_park_count_1500m          0
nearest_poi_park_m            0
poi_pharmacy_count_1500m      0
nearest_poi_pharmacy_m        0
poi_hospital_count_1500m      0
nearest_poi_hospital_m        0
poi_mall_count_1500m          0
nearest_poi_mall_m            0
building_count_1500m          0
residential_ratio_1500m       0
population_1500m              0
signals_1500m                 0
unique_users_1500m            0
dtype: int64


In [21]:
features_df.to_csv('../data/processed/features.csv', index=False)
print(f"Saved features: {features_df.shape}")

Saved features: (50, 29)


## Summary

Feature engineering produced a training-ready dataset of 50 locations × 29 columns
(2 coordinates + target + 25 features), with no missing values.

A radius of 1500m was chosen after initial analysis at 500m yielded near-zero
variance across most categories (e.g. 0 competitors within 500m for 46/50
locations). 1500m provides meaningful signal while remaining within a realistic
catchment area for a grocery store. Building density is represented as a
residential ratio — share of residential buildings (single-family, two-family,
multifamily and collective housing) out of all buildings in radius — rather
than raw count, which better captures neighborhood character. Footfall features
are cached to `data/processed/footfall_features.parquet` to avoid recomputing
50 × ST_DWITHIN Snowflake queries (~13 min) on every notebook run.

This is the **initial version** of the feature set. Several aspects
may change after model training and evaluation in Task 3:

- Features with near-zero importance (e.g. sparse POI categories
  like `poi_bus_stop_count_1500m`, `poi_mall_count_1500m`)
  may be dropped.
- Radius may be revisited — different features may benefit
  from different catchment areas.
- Additional features (e.g. district-level encoding,
  building area density, household size) may be added
  in subsequent iterations based on model performance.
- Correlation between features (e.g. `population_1500m`
  and `signals_1500m`) will be examined in Task 3 —
  highly correlated features may be consolidated.

### Second iteration

Two more features will be engineered: `store_building_area`, which indicates the area of building in which the store is located, and `is_retail_building`, a binary flag which indicates if that building play a retail function

In [22]:
locs_proj = gdf_locations.to_crs('EPSG:2180')
buildings_proj = gdf_buildings.to_crs('EPSG:2180')

store_building = gpd.sjoin(
    locs_proj[['location_id', 'geometry']],
    buildings_proj[['geometry', 'area', 'funogolnabudynku_desc']],
    predicate='within',
    how='left'
)[['location_id', 'area', 'funogolnabudynku_desc']]

store_building = store_building.rename(columns={
    'area': 'store_building_area',
    'funogolnabudynku_desc': 'store_building_type'
})

# If a building is not found for a location, the nearest building is taken.
missing_ids = store_building[store_building['store_building_area'].isna()]['location_id']
missing = locs_proj[locs_proj['location_id'].isin(missing_ids)]

nearest_fallback = gpd.sjoin_nearest(
    missing[['location_id', 'geometry']],
    buildings_proj[['geometry', 'area', 'funogolnabudynku_desc']],
    how='left',
    distance_col='dist'
)[['location_id', 'area', 'funogolnabudynku_desc']]

store_building = store_building.set_index('location_id')
for _, row in nearest_fallback.iterrows():
    if pd.notna(row['area']):
        store_building.loc[row['location_id'], 'store_building_area'] = row['area']
        store_building.loc[row['location_id'], 'store_building_type'] = row['funogolnabudynku_desc']
store_building = store_building.reset_index()

store_building['is_retail_building'] = (
    store_building['store_building_type'] == 'budynkiHandlowoUslugowe'
).astype(int)

In [23]:
store_building

,location_id,store_building_area,store_building_type,is_retail_building
0,LOC_001,1187.77,budynkiHandlowoUslugowe,1
1,LOC_002,1218.72,budynkiBiurowe,0
2,LOC_003,4530.20,budynkiHandlowoUslugowe,1
3,LOC_004,94.78,budynkiMieszkalneJednorodzinne,0
4,LOC_005,1149.73,budynkiHandlowoUslugowe,1
5,LOC_006,1336.61,budynkiHandlowoUslugowe,1
6,LOC_007,932.03,budynkiHandlowoUslugowe,1
7,LOC_008,1170.33,budynkiHoteli,0
8,LOC_009,2931.16,budynkiHandlowoUslugowe,1
9,LOC_010,244.76,budynkiOTrzechIWiecejMieszkaniach,0


In [24]:
features_df = features_df.merge(
    store_building[['location_id', 'store_building_area', 'is_retail_building']],
    on='location_id',
    how='left'
)

In [25]:
features_df.to_csv('../data/processed/features_2.csv', index=False)
print(f"Saved features: {features_df.shape}")

Saved features: (50, 31)


### Third iteration (see notebook 03)

A third feature was explored during model training:
`signals_per_user = signals_1500m / unique_users_1500m` —
a proxy for visit frequency per unique visitor.
It was evaluated in `03_model_and_interpretation.ipynb`
alongside the model training process.
Result: marginal R² improvement (0.180 vs 0.175),
not statistically meaningful at n=50.